# Emotional CMR (eCMR)

CMR with emotional memory channel and LPP modulation


Emotional CMR (eCMR) extends the standard CMR framework to model **emotional memory enhancement**. It incorporates a separate emotional memory channel and allows EEG signals (LPP) to modulate encoding strength.

## The Mechanism

### Dual Memory Channels

eCMR maintains two memory systems:

1. **Standard MCF**: Context → Item associations (as in regular CMR)
2. **Emotional MCF**: Captures emotion-to-emotion clustering during retrieval

At retrieval, if the last recalled item was emotional, the emotional channel provides additional activation support to other emotional items.

### LPP Modulation

The Late Positive Potential (LPP) modulates encoding strength through:

**Main effect**: LPP affects all items (emotional and neutral):
$$\kappa_L (L_i - \lambda_L)$$

**Interaction effect**: LPP specifically affects emotional items:
$$\kappa_{EL} (L_i - \lambda_{EL}) \cdot e_i$$

## Mathematical Specification

### Encoding Phase

When item $i$ is studied, the emotional memory strength is:

$$g^{emot}_i = \max(0, \kappa_E \cdot e_i + \kappa_L (L_i - \lambda_L) + \kappa_{EL} (L_i - \lambda_{EL}) \cdot e_i)$$

The emotional MCF learning rate is either multiplicative or additive with primacy:

**Multiplicative:**
$$g_i = \phi_i \cdot g^{emot}_i$$

**Additive:**
$$g_i = g^{emot}_i$$

The emotional memory channel is updated:

In [ ]:
emotional_mcf = emotional_mcf.at[item_index].multiply(emcf_lr)

### Retrieval Phase

Activations combine standard MCF and emotional channel:

In [ ]:
def activations(self):
    # Standard context-to-item support
    _activations = self.mcf.probe(self.context.state) * self.recallable

    # Emotional support (only if last recall was emotional)
    last_emotional = lax.cond(
        self.recall_total > 0,
        lambda: self.is_emotional[self.recalls[self.recall_total - 1] - 1] > 0,
        lambda: False,
    )
    emotion_support = last_emotional * self.emotional_mcf

    # Merge and apply sensitivity
    merged_support = power_scale(
        _activations + emotion_support,
        self.mcf_sensitivity,
    )
    return (merged_support + lb) * self.recallable

## Parameters

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| `emotion_scale` | $\kappa_E$ | Base boost for emotional items |
| `lpp_main_scale` | $\kappa_L$ | LPP main effect slope |
| `lpp_main_threshold` | $\lambda_L$ | LPP main effect threshold |
| `lpp_inter_scale` | $\kappa_{EL}$ | LPP×emotion interaction slope |
| `lpp_inter_threshold` | $\lambda_{EL}$ | LPP×emotion interaction threshold |
| `modulate_emotion_by_primacy` | — | Whether to multiply by primacy gradient |

Plus all standard CMR parameters.

## Predictions

### Emotional Clustering

After recalling an emotional item:
- The emotional channel becomes active
- Other emotional items receive a retrieval boost
- Predicted: clustering of emotional items in recall

### LPP-Memory Relationship

Higher LPP during encoding predicts:
- Better memory for that specific item
- Effect can be general (main effect) or emotion-specific (interaction)

### Serial Position Effects

With `modulate_emotion_by_primacy=True`:
- Emotional enhancement is strongest for early items
- Primacy and emotion interact multiplicatively
- Captures finding that emotional primacy effects are enhanced

## Usage

In [ ]:
from jaxcmr.models_eeg.eeg_ecmr import CMR, make_factory
import jaxcmr.components.linear_memory as LinearMemory
import jaxcmr.components.context as TemporalContext
from jaxcmr.components.termination import PositionalTermination

Factory = make_factory(
    LinearMemory.init_mfc,
    LinearMemory.init_mcf,
    TemporalContext.init,
    PositionalTermination,
)

factory = Factory(dataset, features=None)

params = {
    # Standard CMR
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "learning_rate": 0.5,
    "primacy_scale": 2.0,
    "primacy_decay": 0.8,
    "shared_support": 0.05,
    "item_support": 0.25,
    "choice_sensitivity": 1.0,

    # Emotional parameters
    "emotion_scale": 1.0,
    "lpp_main_scale": 0.5,
    "lpp_main_threshold": 0.0,
    "lpp_inter_scale": 0.3,
    "lpp_inter_threshold": 0.0,
    "modulate_emotion_by_primacy": True,

    # Other
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "learn_after_context_update": True,
    "allow_repeated_recalls": False,
}

model = factory.create_trial_model(trial_index=0, parameters=params)

## EEG-CMR (Main Effects) {#eeg-cmr-main-effects}

A more parsimonious variant removes the separate emotional memory channel. Instead, emotion and LPP modulate the **standard MCF** directly.

### Key Difference

In eCMR: Two memory channels (standard + emotional)
In EEG-CMR: One memory channel with modulated learning rate

### Encoding in EEG-CMR

The encoding modulation combines all effects:

In [ ]:
encoding_modulation = (
    self.emotion_scale * self.is_emotional
    + self.lpp_main
    + self.lpp_interaction
)

This modulates the MCF learning rate directly:

In [ ]:
def _multiplicative():
    return primacy * modulation_clipped

def _additive():
    return primacy + jnp.maximum(-primacy, modulation_clipped)

mcf_lr = lax.cond(self.modulate_emotion_by_primacy, _multiplicative, _additive)

### When to Use Which

**Use eCMR when:**
- You expect emotional clustering at retrieval
- The emotional channel is theoretically motivated
- Data shows emotion-to-emotion transitions

**Use EEG-CMR when:**
- You want a simpler model
- Encoding effects are the primary interest
- Testing whether retrieval-phase emotion effects are necessary

## Exponent Parameters

The EEG-CMR variant includes optional exponent parameters for nonlinear LPP transforms:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `lpp_main_exponent` | 1.0 | Exponent for main effect term |
| `lpp_inter_exponent` | 1.0 | Exponent for interaction term |

With exponent = 1.0 (default), the relationship is linear.
With exponent > 1.0, extreme LPP values have disproportionate effects.
With exponent < 1.0, the effect compresses toward the center.

## Model Comparison

| Aspect | eCMR | EEG-CMR |
|--------|------|---------|
| Memory channels | 2 | 1 |
| Retrieval-phase emotion | Yes | No |
| Emotional clustering | Predicted | Not predicted |
| Parameters | More | Fewer |
| Theoretical commitment | Separate emotion system | Unified encoding |

## Theoretical Background

The eCMR model builds on:

- **Talmi et al. (2019)**: Resource economy account of emotional memory
- **Polyn et al. (2009)**: Source-memory CMR with category channels
- **Subsequent memory paradigm**: Using encoding activity to predict later recall

The key insight is that emotional processing during encoding—indexed by the LPP—creates stronger memory traces that are preferentially retrieved.